In [ ]:
# Scenario: AI Symptom Tracker (Question-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:
# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"
# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.
# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.
# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.
# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.
# - End Node
# - Delivers the final recommendation to the patient.

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# 1. State Definition
class SymptomState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int

# 2. Observation Node (LLM)
def generate_observation(state: SymptomState):
    api_key = os.getenv("NVIDIA_API_KEY")

    prompt = f"""
    A patient reports: {state['symptom']}

    Give ONE possible general observation (not diagnosis).
    Keep it short and simple.
    """

    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta/llama3-8b-instruct",
            "messages": [
                {"role": "system", "content": "You are a safe health assistant. Do NOT give diagnosis."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.7
        }
    )

    data = response.json()
    obs = data["choices"][0]["message"]["content"].strip()

    # avoid duplicates
    if obs in state["observations"]:
        obs = "General lifestyle or environmental factor may contribute."

    return {
        "observations": state["observations"] + [obs],
        "steps_done": state["steps_done"] + 1
    }

# 3. Analysis Node
def analyze(state: SymptomState):
    combined = " | ".join(state["observations"])

    analysis = f"Possible factors related to '{state['symptom']}': {combined}"

    return {"analysis": analysis}

# 4. Conditional Check
def check_condition(state: SymptomState):
    if len(state["observations"]) < 3:
        return "observe"
    else:
        return "recommend"

# 5. Recommendation Node
def generate_recommendation(state: SymptomState):
    api_key = os.getenv("NVIDIA_API_KEY")

    prompt = f"""
    Symptom: {state['symptom']}
    Observations: {state['observations']}

    Give a simple, safe, non-medical recommendation.
    Do NOT diagnose.
    """

    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta/llama3-8b-instruct",
            "messages": [
                {"role": "system", "content": "You are a safe assistant. Avoid medical diagnosis."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.5
        }
    )

    data = response.json()
    rec = data["choices"][0]["message"]["content"].strip()

    return {"recommendation": rec}

# 6. Output Node
def show_result(state: SymptomState):
    print("\n📝 Observations:")
    for i, obs in enumerate(state["observations"], 1):
        print(f"{i}. {obs}")

    print("\n🔍 Analysis:")
    print(state["analysis"])

    print("\n💡 Recommendation:")
    print(state["recommendation"])

    return {}

# 7. Build Graph
graph = StateGraph(SymptomState)

graph.add_node("observe", generate_observation)
graph.add_node("analyze", analyze)
graph.add_node("recommend", generate_recommendation)
graph.add_node("output", show_result)

graph.set_entry_point("observe")

# Loop until 3 observations
graph.add_conditional_edges(
    "observe",
    check_condition,
    {
        "observe": "observe",
        "recommend": "analyze"
    }
)

graph.add_edge("analyze", "recommend")
graph.add_edge("recommend", "output")
graph.add_edge("output", END)

# 8. Run
if __name__ == "__main__":
    app = graph.compile()

    state = {
        "symptom": input("Enter symptom: "),
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0
    }

    app.invoke(state)


📝 Observations:
1. The patient appears to be experiencing discomfort or distress.
2. The patient's temperature is elevated.
3. The patient appears uncomfortable and slightly fatigued.

🔍 Analysis:
Possible factors related to 'fever': The patient appears to be experiencing discomfort or distress. | The patient's temperature is elevated. | The patient appears uncomfortable and slightly fatigued.

💡 Recommendation:
Based on the symptoms and observations, I would recommend the following simple and safe non-medical action:

* Encourage the patient to rest and stay hydrated by drinking plenty of fluids, such as water or clear broth.

This recommendation aims to help alleviate the patient's discomfort and distress, while avoiding any medical diagnosis or treatment.
